# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**The question:** which pages should an editor look at first? That is a *ranking* question with a yes/no proxy label (`is_declining`, from Week 4), so the toolkit points to **Logistic Regression first, then stronger models as a check**.

- **Logistic Regression:** the readable starting point. It gives one weight per feature, so I can say which way each one pushes.
- **Random Forest and Gradient Boosting:** the stronger models. They only earn a place if they beat the simple one by more than the noise. Gradient boosting is safe here because the split is grouped by client and the leakage guard below passes.
- **Decision Tree (depth 3):** a tree I could print and read, as a check on whether a few simple rules do the job.
- **The Week-4 rule:** the baseline every model has to beat, scored on the same data.

I skipped clustering because my question is not about grouping pages, and the signal analysis was already Week 4's job. To see what the chosen model leans on, I use **permutation importance** and an ablation in section 4.

In [1]:
import json, os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
import sklearn

SEED = 42
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)   # the label: evaluation and training target only

for c in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df["log_" + c] = np.log1p(df[c])

NUM = ["search_volume", "competition", "cpc", "word_count", "char_count",
       "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
       "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
       "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"]
CAT = ["competition_level", "content_type", "main_intent"]

# Leakage guard: no label column, no label ingredient, no recent-window column may be a feature.
FORBIDDEN = {"trend_direction", "trend_pct", "is_declining"} | {c for c in df.columns if c.endswith("_last_30d") or c.endswith("_prev_30d")}
assert not (set(NUM + CAT) & FORBIDDEN), "a forbidden column is in the feature list"

print(f"rows: {len(df):,} | clients: {df['client_id'].nunique()} | base rate of the declining proxy: {df['is_declining'].mean():.3f}")
print(f"features: {len(NUM)} numeric + {len(CAT)} categorical | leakage guard passed")
print(f"python libs: pandas {pd.__version__}, scikit-learn {sklearn.__version__} | seed {SEED}")

rows: 30,000 | clients: 32 | base rate of the declining proxy: 0.542
features: 18 numeric + 3 categorical | leakage guard passed
python libs: pandas 3.0.6, scikit-learn 1.9.1 | seed 42


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split: 5-fold, grouped by client.** Every page from one client sits on one side of the split, so each test page is scored by a model that never saw its client.

- **Why this is honest for my question:** an editor asks about a *new* page, usually for a client the model hasn't seen. A random row split would let the model learn a client's habits and look better than it is.
- **Why not a time-aware split:** the file is one snapshot with no history, so I can't test forward in time. That limit is named in section 4.
- **Same split for the baseline:** the Week-4 rule has nothing to train, so it is scored on the same five test folds as the models.
- **Reproducibility:** seed 42 everywhere. `GroupKFold` itself has no randomness.

Fold 1 is a single client (7,008 pages) because `GroupKFold` balances by size, so fold sizes are uneven and the test base rate ranges from 0.38 to 0.65.

In [2]:
X = df[NUM + CAT]
y = df["is_declining"].values
groups = df["client_id"].values

folds = list(GroupKFold(n_splits=5).split(X, y, groups))

rows = []
for i, (tr, te) in enumerate(folds, 1):
    assert not (set(groups[tr]) & set(groups[te])), "a client appears on both sides"
    rows.append({"fold": i, "train_rows": len(tr), "test_rows": len(te),
                 "test_clients": len(set(groups[te])), "test_base_rate": round(y[te].mean(), 3)})
pd.DataFrame(rows)

,fold,train_rows,test_rows,test_clients,test_base_rate
0,1,22992,7008,1,0.490
1,2,24269,5731,7,0.645
2,3,24247,5753,8,0.379
3,4,24245,5755,8,0.622
4,5,24247,5753,8,0.585


Client declining rates run from **0% to 94%** (median about 52%). A model that doesn't know the client can't see that shift, and it will come back as the main error in section 4.

In [3]:
client_rates = (df.groupby("client_id")["is_declining"].agg(pages="size", declining_rate="mean").round(3))
print("declining rate per client: min", client_rates["declining_rate"].min(), "| median", client_rates["declining_rate"].median(), "| max", client_rates["declining_rate"].max())
client_rates.sort_values("pages", ascending=False).head(6)

declining rate per client: min 0.0 | median 0.524 | max 0.937


,pages,declining_rate
client_id,,
client_19581e27de,7008,0.490
client_6208ef0f77,3681,0.644
client_4e07408562,2294,0.493
client_3fdba35f04,2267,0.839
client_f369cb89fc,1796,0.601
client_8527a891e2,1194,0.437


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Same data, same folds, same metrics as the Week-4 rule.** I report three numbers, each averaged over the five test folds:

- **ROC AUC:** how well the whole list of 30,000 pages is ordered (0.5 is random).
- **precision@100 and precision@500:** how many of the top 100 or 500 are really in the declining proxy. This is what an editor's queue looks like.

The rule is scored as a continuous number with its original tie-breaks (lowest CTR first, then longest since update). Its P@100 here (0.81) is not the 0.86 from Week 4, because Week 4 scored the top 100 of all 30,000 pages and this scores the top 100 of each ~6,000-page test fold.

In [4]:
# --- The Week-4 rule, unchanged, as a continuous score so it can be ranked and scored like a model ---
vis  = df["impressions_90d"] >= 500
p1   = (df["avg_position"] > 0) & (df["avg_position"] <= 10)
gap  = (vis & p1 & (df["ctr"] <= 0.12)).astype(int)
stale = (vis & (df["days_since_last_update"] >= 90)).astype(int)
points = 2 * gap + stale
# ties broken exactly as in Week 4: lowest CTR first, then longest since last update
df["rule_score"] = points * 1000 + (6 - df["ctr"].clip(upper=5.5)) + df["days_since_last_update"].clip(upper=3650) / 1e5

def preprocess(scale):
    num = (make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler())
           if scale else SimpleImputer(strategy="median", add_indicator=True))
    cat = make_pipeline(SimpleImputer(strategy="constant", fill_value="missing"), OneHotEncoder(handle_unknown="ignore"))
    return ColumnTransformer([("num", num, NUM), ("cat", cat, CAT)])

def make_models():
    return {
        "Logistic Regression": make_pipeline(preprocess(True),  LogisticRegression(max_iter=3000)),
        "Decision Tree (depth 3)": make_pipeline(preprocess(False), DecisionTreeClassifier(max_depth=3, random_state=SEED)),
        "Random Forest": make_pipeline(preprocess(False), RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=20, random_state=SEED, n_jobs=-1)),
        "Gradient Boosting": make_pipeline(preprocess(False), HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=200, random_state=SEED)),
    }

def precision_at_k(y_true, score, k):
    order = np.lexsort((np.arange(len(score)), -score))[:k]     # highest score first, stable
    return y_true[order].mean()

KS = (100, 500)
per_fold = {"Week-4 rule": []}
oof = {"Week-4 rule": df["rule_score"].values.astype(float)}
for name in make_models():
    per_fold[name] = []
    oof[name] = np.zeros(len(df))

for i, (tr, te) in enumerate(folds):
    yt = y[te]
    per_fold["Week-4 rule"].append((roc_auc_score(yt, oof["Week-4 rule"][te]), *[precision_at_k(yt, oof["Week-4 rule"][te], k) for k in KS], yt.mean()))
    for name, model in make_models().items():
        model.fit(X.iloc[tr], y[tr])
        s = model.predict_proba(X.iloc[te])[:, 1]
        oof[name][te] = s
        per_fold[name].append((roc_auc_score(yt, s), *[precision_at_k(yt, s, k) for k in KS], yt.mean()))

rows = []
for name, vals in per_fold.items():
    a = np.array(vals)
    rows.append({"method": name,
                 "ROC AUC": f"{a[:,0].mean():.3f} +/- {a[:,0].std():.3f}",
                 "precision@100": round(a[:,1].mean(), 3),
                 "precision@500": round(a[:,2].mean(), 3),
                 "lift@100 vs base": round(a[:,1].mean() / a[:,3].mean(), 2)})
table = pd.DataFrame(rows)
base_rate_folds = np.mean([v[3] for v in per_fold["Week-4 rule"]])
print(f"base rate (random picking), mean of the 5 test folds: {base_rate_folds:.3f}")
table

base rate (random picking), mean of the 5 test folds: 0.544


,method,ROC AUC,precision@100,precision@500,lift@100 vs base
0,Week-4 rule,0.526 +/- 0.048,0.810,0.729,1.49
1,Logistic Regression,0.658 +/- 0.030,0.800,0.747,1.47
2,Decision Tree (depth 3),0.611 +/- 0.068,0.602,0.616,1.11
3,Random Forest,0.668 +/- 0.036,0.682,0.736,1.25
4,Gradient Boosting,0.677 +/- 0.042,0.776,0.738,1.43


In [5]:
fold_auc = pd.DataFrame({name: [round(v[0], 3) for v in vals] for name, vals in per_fold.items()}, index=[f"fold {i}" for i in range(1, 6)])
fold_auc

,Week-4 rule,Logistic Regression,Decision Tree (depth 3),Random Forest,Gradient Boosting
fold 1,0.565,0.618,0.585,0.657,0.647
fold 2,0.556,0.631,0.497,0.625,0.622
fold 3,0.503,0.701,0.697,0.735,0.747
fold 4,0.565,0.677,0.654,0.657,0.682
fold 5,0.443,0.663,0.621,0.664,0.688


### Model vs baseline

| Method | ROC AUC (mean +/- std over 5 folds) | precision@100 | precision@500 |
|---|---|---|---|
| Random picking (base rate) | 0.50 | 0.544 | 0.544 |
| **Week-4 rule** | 0.526 +/- 0.048 | 0.810 | 0.729 |
| **Logistic Regression** | 0.658 +/- 0.030 | 0.800 | 0.747 |
| Decision Tree (depth 3) | 0.611 +/- 0.068 | 0.602 | 0.616 |
| Random Forest | 0.668 +/- 0.036 | 0.682 | 0.736 |
| Gradient Boosting | 0.677 +/- 0.042 | 0.776 | 0.738 |

**What the table says:**

- **The rule is good at the top, weak everywhere else.** Its AUC is close to 0.5 because it gives zero points to about 74% of pages, which all tie. But its top 100 is 81% declining.
- **Logistic Regression orders the whole list far better.** It beats the rule on AUC in all five folds (about +0.13). At the top of the list it is level with the rule (0.80 vs 0.81 at P@100), and slightly ahead at P@500 (0.747 vs 0.729), which is within noise.
- **Complexity did not pay.** Gradient Boosting is +0.019 AUC over Logistic Regression (it wins 4 of 5 folds), but the gap is smaller than the fold-to-fold spread and its P@100 is lower (0.776). Random Forest has a similar AUC but a P@100 of only 0.682, worse than the rule. The depth-3 tree is the weakest.
- **My pick: Logistic Regression.** It is the simplest, it is readable, and it is within noise of the best AUC while having the best top-100 of the models.

With 32 clients and 5 folds, treat gaps of about 0.02 as noise.

In [6]:
metrics = {
    "task": "w05_model",
    "split": "GroupKFold(5) by client_id, no client on both sides",
    "seed": SEED,
    "base_rate_mean_test_folds": round(float(base_rate_folds), 4),
    "results": {r["method"]: {"roc_auc": r["ROC AUC"], "precision_at_100": float(r["precision@100"]),
                               "precision_at_500": float(r["precision@500"])} for r in rows},
    "features_numeric": NUM, "features_categorical": CAT,
}
os.makedirs("../outputs", exist_ok=True)
with open("../outputs/w05_model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("wrote ../outputs/w05_model_metrics.json")

wrote ../outputs/w05_model_metrics.json


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Three questions: **what does the chosen model lean on, is that suspicious, and where is it wrong?** I use permutation importance (shuffle one feature on held-out clients and see how much AUC drops), the direction of the coefficients, and an ablation.

In [7]:
# What does the chosen model lean on? Permutation importance on HELD-OUT clients, averaged over the 5 folds.
chosen = "Logistic Regression"
imp = pd.DataFrame(index=NUM + CAT, columns=range(5), dtype=float)
for i, (tr, te) in enumerate(folds):
    m = make_models()[chosen].fit(X.iloc[tr], y[tr])
    r = permutation_importance(m, X.iloc[te], y[te], scoring="roc_auc", n_repeats=5, random_state=SEED)
    imp[i] = r.importances_mean
perm = pd.DataFrame({"mean AUC drop when shuffled": imp.mean(axis=1).round(4), "folds where it hurts": (imp > 0).sum(axis=1)})
perm.sort_values("mean AUC drop when shuffled", ascending=False).head(8)

,mean AUC drop when shuffled,folds where it hurts
log_impressions_90d,0.1565,5
log_clicks_90d,0.0584,4
char_count,0.0245,3
word_count,0.0219,4
log_sessions_90d,0.0153,3
avg_position,0.0138,4
content_age_days,0.0121,4
content_type,0.0050,4


In [8]:
# Direction of the effects: standardised coefficients from one fit on all data (for reading only, never scored).
m_all = make_models()[chosen].fit(X, y)
names = m_all[:-1].get_feature_names_out()
coef = pd.Series(m_all[-1].coef_[0], index=names).sort_values()
print("pushes the prediction toward 'not declining':"); print(coef.head(5).round(3).to_string())
print("\npushes the prediction toward 'declining':");     print(coef.tail(5).round(3).to_string())

pushes the prediction toward 'not declining':
num__log_clicks_90d             -0.739
cat__main_intent_navigational   -0.414
num__content_age_days           -0.272
cat__competition_level_LOW      -0.264
num__avg_position               -0.259

pushes the prediction toward 'declining':
cat__content_type_keyword article    0.209
cat__main_intent_informational       0.211
num__word_count                      0.264
cat__competition_level_missing       0.493
num__log_impressions_90d             1.184


In [9]:
# Ablation: is the model leaning on the 90-day totals that overlap the label's 30-day windows?
VOLUME = ["log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d"]
CONTENT_ONLY = ["search_volume", "competition", "cpc", "word_count", "char_count", "content_age_days", "days_since_last_update"]

def logreg_auc(num_cols):
    aucs = []
    for tr, te in folds:
        pre = ColumnTransformer([
            ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), num_cols),
            ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="missing"), OneHotEncoder(handle_unknown="ignore")), CAT)])
        m = make_pipeline(pre, LogisticRegression(max_iter=3000)).fit(X.iloc[tr], y[tr])
        aucs.append(roc_auc_score(y[te], m.predict_proba(X.iloc[te])[:, 1]))
    return f"{np.mean(aucs):.3f} +/- {np.std(aucs):.3f}"

pd.Series({
    "all features":                                   logreg_auc(NUM),
    "without log_impressions_90d":                    logreg_auc([c for c in NUM if c != "log_impressions_90d"]),
    "without the four 90-day volume columns":         logreg_auc([c for c in NUM if c not in VOLUME]),
    "content and age only (no performance columns)":  logreg_auc(CONTENT_ONLY),
}, name="ROC AUC, logistic regression, same folds").to_frame()

,"ROC AUC, logistic regression, same folds"
all features,0.658 +/- 0.030
without log_impressions_90d,0.631 +/- 0.036
without the four 90-day volume columns,0.637 +/- 0.031
content and age only (no performance columns),0.532 +/- 0.035


**What it leans on.** By permutation importance the top three are `log_impressions_90d` (AUC drops 0.157 when shuffled, in all 5 folds), `log_clicks_90d` (0.058) and `char_count` (0.025). In the coefficients, more impressions pushes toward *declining* and more clicks pushes away from it. Together that reads as "lots of impressions but few clicks", which matches the low-CTR signal I confirmed in Week 4. Navigational intent and older pages push away from declining.

**Is it suspicious?** The overall AUC is 0.658, so nothing is suspiciously perfect. But the top features are 90-day totals, and those 90 days include the two 30-day windows the label is built from. The ablation checks this: without `log_impressions_90d` the AUC falls from 0.658 to 0.631, and without all four volume columns to 0.637. With no performance columns at all (content and age only) it falls to 0.532, close to random.

**What that means:** no single leaked column is carrying the model. But nearly all of its signal comes from 90-day performance numbers that partly overlap the label window. So it is better described as recognising pages that are *currently* falling than as forecasting which pages *will* fall.

In [10]:
# Error analysis on out-of-fold predictions: every page is scored by a model that never saw its client.
df["pred"] = oof[chosen]

df["pred_decile"] = pd.qcut(df["pred"].rank(method="first"), 10, labels=False) + 1
calib = df.groupby("pred_decile").agg(pages=("pred", "size"), mean_predicted=("pred", "mean"), actual_declining=("is_declining", "mean")).round(3)
calib

,pages,mean_predicted,actual_declining
pred_decile,,,
1,3000,0.246,0.295
2,3000,0.370,0.355
3,3000,0.447,0.469
4,3000,0.507,0.513
5,3000,0.559,0.556
6,3000,0.606,0.599
7,3000,0.652,0.610
8,3000,0.699,0.646
9,3000,0.750,0.666


In [11]:
by_client = (df.groupby("client_id")
               .apply(lambda g: pd.Series({"pages": len(g), "actual_rate": g["is_declining"].mean(), "mean_predicted": g["pred"].mean(),
                                           "auc": roc_auc_score(g["is_declining"], g["pred"]) if g["is_declining"].nunique() == 2 and len(g) >= 200 else np.nan}))
               .round(3))
by_client["gap (predicted - actual)"] = (by_client["mean_predicted"] - by_client["actual_rate"]).round(3)
by_client["client"] = by_client.index.str[-4:]
big = by_client[by_client["pages"] >= 200].set_index("client")
print("clients with 200+ pages:", len(big), "| median per-client AUC:", round(big["auc"].median(), 3), "| clients with AUC below 0.5:", int((big["auc"] < 0.5).sum()))
print("\nlargest prediction gaps:")
big.reindex(big["gap (predicted - actual)"].abs().sort_values(ascending=False).index).head(5)[["pages", "actual_rate", "mean_predicted", "auc", "gap (predicted - actual)"]]

clients with 200+ pages: 22 | median per-client AUC: 0.618 | clients with AUC below 0.5: 2

largest prediction gaps:


,pages,actual_rate,mean_predicted,auc,gap (predicted - actual)
client,,,,,
7096,476.0,0.000,0.352,NaN,0.352
5f04,2267.0,0.839,0.555,0.635,-0.284
d7e2,1043.0,0.937,0.667,0.580,-0.270
6ff0,254.0,0.386,0.642,0.523,0.256
9fc2,556.0,0.595,0.348,0.594,-0.247


In [12]:
# Three concrete wrong cases: the highest-scored pages that were NOT declining.
cols = ["pred", "impressions_90d", "avg_position", "ctr", "days_since_last_update", "content_age_days", "main_intent", "trend_direction"]
wrong = df[df["is_declining"] == 0].sort_values("pred", ascending=False).head(3)[cols].copy()
wrong.insert(0, "client", df.loc[wrong.index, "client_id"].str[-4:])
wrong["pred"] = wrong["pred"].round(3)
wrong

,client,pred,impressions_90d,avg_position,ctr,days_since_last_update,content_age_days,main_intent,trend_direction
12869,8562,0.957,15101,5.7,0.00,7,421,informational,stable
13020,27de,0.955,19360,6.9,0.01,104,236,informational,stable
21657,27de,0.953,3881,8.5,0.00,104,144,informational,up


**Where it is wrong:**

- **Calibration is good in the middle and overconfident at the ends.** The ordering is right, and the actual declining rate climbs from 0.30 in the lowest-scored decile to 0.71 in the highest. But the top decile is predicted at 0.83 and is actually 0.71.
- **The biggest error is the client.** Client declining rates run from 0% to 94%, and the model can't know which client is in decline. One client (476 pages) has no declining pages at all, yet the model predicts 0.35 on average. Two others are 0.84 and 0.94 declining against predictions of 0.56 and 0.67. Inside a client the ranking is decent (median per-client AUC 0.618), but 2 of 22 larger clients are below 0.5.
- **Three concrete misses.** The three highest-scored pages that were not declining are all large (3,881 to 19,360 impressions), sit at positions 5.7 to 8.5, have a CTR near zero, and are informational. Two were stable and one was trending up. They fit the pattern the model learned, but a big page with a low CTR can stay steady for months. The model sees 90-day totals and not the recent trend, and one of the three was updated only 7 days earlier.

**Limits:** one snapshot, so no forward-in-time test. The label is a proxy for "declining now", not "worth a refresh". The 90-day inputs partly overlap the label window. 32 clients is a small number to generalise from.

**Bottom line:** a simple Logistic Regression ranks the full list much better than my Week-4 rule (AUC 0.66 vs 0.53) and matches it at the very top. The stronger models don't earn their complexity. The honest claim is decision-support for ordering a review queue, not a forecast. The next test is the sealed final month in the warehouse.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.